# NB08 — Robustness Evaluation
Loads saved model checkpoints from NB05, runs ensemble inference on all held-out test splits.
**Requires GPU** (~20 min). Run after NB05 and NB06.

In [1]:
import os, json, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    f1_score, accuracy_score, matthews_corrcoef,
    classification_report, roc_auc_score
)

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

os.makedirs("../outputs/robustness", exist_ok=True)


Device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


## Config — matches NB05 exactly

In [2]:
CONFIG = {
    "models": {
        "banglabert": "csebuetnlp/banglabert",
        "muril":      "google/muril-base-cased",
        "xlmr":       "xlm-roberta-base",
    },
    "max_length": 128,
    "batch_size": 64,   # inference only — can be larger than training
}

TASK_CONFIG = {
    "binary": {
        "col": "label_binary",
        "num_classes": 2,
    },
    "abuse_type": {
        "col": "label_type",
        "num_classes": None,
    },
}

MODELS_DIR = "../outputs/models"
SPLITS_DIR = "../data/splits"


## Reconstruct label encoders (must match NB05)

In [3]:
# Load the same splits NB05 used to reconstruct identical label_encoders
train_df = pd.read_csv(f"{SPLITS_DIR}/random_train.csv")
val_df   = pd.read_csv(f"{SPLITS_DIR}/random_val.csv")
test_df  = pd.read_csv(f"{SPLITS_DIR}/random_test.csv")

label_encoders = {}
active_tasks   = {}

for task_name, task_cfg in TASK_CONFIG.items():
    col = task_cfg["col"]
    if col in train_df.columns:
        all_vals = pd.concat([train_df[col], val_df[col], test_df[col]]).dropna().unique()
        label_map = {v: i for i, v in enumerate(sorted(all_vals))}
        label_encoders[task_name] = label_map
        task_cfg["num_classes"] = len(label_map)
        active_tasks[task_name] = task_cfg
        print(f"Task '{task_name}': {len(label_map)} classes")
    else:
        print(f"Task '{task_name}': column '{col}' not found — SKIPPED")

print(f"\nActive tasks: {list(active_tasks.keys())}")


Task 'binary': 2 classes
Task 'abuse_type': 89 classes

Active tasks: ['binary', 'abuse_type']


## Model architecture (copy from NB05)

In [4]:
class CyberBullyDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, active_tasks, label_encoders):
        self.texts = df["text_clean"].fillna("").tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = {}
        for task_name, task_cfg in active_tasks.items():
            col = task_cfg["col"]
            enc = label_encoders[task_name]
            self.labels[task_name] = [
                enc.get(str(v) if not pd.isna(v) else "unknown", -1)
                for v in df[col].fillna("unknown")
            ]

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        for task_name in self.labels:
            item[f"label_{task_name}"] = torch.tensor(
                self.labels[task_name][idx], dtype=torch.long
            )
        return item


class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, active_tasks, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size
        self.heads    = nn.ModuleDict()
        self.dropouts = nn.ModuleDict()
        for task_name, task_cfg in active_tasks.items():
            self.dropouts[task_name] = nn.Dropout(dropout)
            self.heads[task_name]    = nn.Linear(hidden_size, task_cfg["num_classes"])

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        outputs    = self.encoder(**kwargs)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return {t: self.heads[t](self.dropouts[t](cls_output)) for t in self.heads}


## Load all saved checkpoints

In [5]:
# Discover checkpoints
ckpt_dirs = sorted(glob.glob(f"{MODELS_DIR}/*_seed*"))
print(f"Found {len(ckpt_dirs)} checkpoints")

# Group by model key
from collections import defaultdict
ckpts_by_model = defaultdict(list)
for d in ckpt_dirs:
    name = os.path.basename(d)
    model_key = "_".join(name.split("_")[:-1])   # strip _seedN
    ckpt_path = os.path.join(d, "best_model.pt")
    if os.path.exists(ckpt_path):
        ckpts_by_model[model_key].append((name, ckpt_path))
        print(f"  ✅ {name}")
    else:
        print(f"  ⚠  Missing checkpoint: {name}")


Found 9 checkpoints
  ✅ banglabert_seed123
  ✅ banglabert_seed42
  ✅ banglabert_seed456
  ✅ muril_seed123
  ✅ muril_seed42
  ✅ muril_seed456
  ✅ xlmr_seed123
  ✅ xlmr_seed42
  ✅ xlmr_seed456


## Inference helper

In [6]:
@torch.no_grad()
def get_binary_logits(model, dataloader):
    model.eval()
    all_logits = []
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch.get("token_type_ids")
        )
        all_logits.append(out["binary"].cpu())
    return torch.cat(all_logits, dim=0)  # (N, 2)


def run_ensemble_on_split(split_df, ensemble_logits_list):
    """Average logits across all runs and return probs + preds."""
    stacked  = torch.stack(ensemble_logits_list, dim=0)   # (K, N, 2)
    avg      = stacked.mean(dim=0)                         # (N, 2)
    probs    = F.softmax(avg, dim=-1)[:, 1].numpy()        # prob of Harmful
    preds    = (probs >= 0.5).astype(int)                  # default threshold
    y_true   = split_df["label_binary"].values
    return y_true, preds, probs


def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "macro_f1":    round(f1_score(y_true, y_pred, average="macro"), 4),
        "weighted_f1": round(f1_score(y_true, y_pred, average="weighted"), 4),
        "mcc":         round(matthews_corrcoef(y_true, y_pred), 4),
        "auroc":       round(roc_auc_score(y_true, y_prob), 4),
    }


## Define held-out test splits to evaluate

In [7]:
ROBUSTNESS_SPLITS = {
    "random_test (in-domain)":           "random_test.csv",
    "source_holdout_banth":              "source_holdout_banth_test.csv",
    "source_holdout_bd_shs":             "source_holdout_bd_shs_test.csv",
    "source_holdout_facebook_44001":     "source_holdout_facebook_44001_test.csv",
    "source_holdout_multilabel_12557":   "source_holdout_multilabel_12557_test.csv",
    "script_holdout_romanized":          "script_holdout_test.csv",
}

# Verify splits exist
for name, fname in ROBUSTNESS_SPLITS.items():
    path = os.path.join(SPLITS_DIR, fname)
    exists = os.path.exists(path)
    n = sum(1 for _ in open(path)) - 1 if exists else 0
    print(f"  {'✅' if exists else '❌'} {name}: {n:,} samples")


UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 373: character maps to <undefined>

## Main loop — run inference on all splits

In [ ]:
all_robustness_results = []

for split_name, split_file in ROBUSTNESS_SPLITS.items():
    split_path = os.path.join(SPLITS_DIR, split_file)
    if not os.path.exists(split_path):
        print(f"⚠ Skipping {split_name} — file not found")
        continue

    split_df = pd.read_csv(split_path)
    print(f"\n{'='*60}")
    print(f"Split: {split_name}  ({len(split_df):,} samples)")
    print(f"{'='*60}")

    ensemble_logits_list = []

    for model_key, model_name in CONFIG["models"].items():
        if model_key not in ckpts_by_model:
            print(f"  ⚠ No checkpoints for {model_key} — skipping")
            continue

        # Load tokenizer once per model_key (reuse across seeds)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        ds = CyberBullyDataset(split_df, tokenizer, CONFIG["max_length"],
                               active_tasks, label_encoders)
        dl = DataLoader(ds, batch_size=CONFIG["batch_size"],
                        shuffle=False, num_workers=0)

        for run_name, ckpt_path in ckpts_by_model[model_key]:
            model = MultiTaskTransformer(model_name, active_tasks).to(device)
            model.load_state_dict(
                torch.load(ckpt_path, map_location=device, weights_only=True)
            )
            logits = get_binary_logits(model, dl)
            ensemble_logits_list.append(logits)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"  ✅ {run_name}")

    if not ensemble_logits_list:
        print("  ❌ No logits collected — skipping")
        continue

    y_true, y_pred, y_prob = run_ensemble_on_split(split_df, ensemble_logits_list)
    metrics = compute_metrics(y_true, y_pred, y_prob)

    print(f"\n  Macro-F1    : {metrics['macro_f1']:.4f}  ← PRIMARY")
    print(f"  Accuracy    : {metrics['accuracy']:.4f}")
    print(f"  Weighted-F1 : {metrics['weighted_f1']:.4f}")
    print(f"  MCC         : {metrics['mcc']:.4f}")
    print(f"  AUROC       : {metrics['auroc']:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Not Harmful','Harmful'], digits=4)}")

    row = {"split": split_name, **metrics}
    all_robustness_results.append(row)

robustness_df = pd.DataFrame(all_robustness_results)
robustness_df.to_csv("../outputs/robustness/robustness_results.csv", index=False)
print("\n✅ Saved: ../outputs/robustness/robustness_results.csv")
print(robustness_df[["split","macro_f1","accuracy","mcc","auroc"]].to_string(index=False))


## Summary table

In [ ]:
print("\n" + "="*70)
print("ROBUSTNESS SUMMARY")
print("="*70)
print(robustness_df[["split","macro_f1","weighted_f1","mcc","auroc"]].to_string(index=False))
